# Stage 2 — real ten-question setup check
**Due 11 October 2026.** Run cells individually. Read `DATA_SETUP.md` and `GPU_STEPS.md` in your separate Stage 2 private folder first.
This notebook collects baseline and retrieval outputs with two samples per observation. Samples measure **consistency, not calibrated confidence**. There is no controller fitting here. No real outputs are bundled.


In [ ]:
from google.colab import drive, files
from pathlib import Path, PurePosixPath
import os, sys, json, zipfile, shutil, subprocess, time, datetime, platform
drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/AdaptiveQA2021_consistency/adaptive-confidence-agent')
print('Persistent working folder:', PROJECT)


In [ ]:
# First session only: upload stage2-colab-source.zip created from your committed local repository.
# Existing source is reused. It is never overwritten by a later upload.
if not (PROJECT/'src/adaptive_qa').exists():
    uploaded=files.upload()
    if len(uploaded)!=1: raise ValueError('Upload exactly one project source ZIP')
    archive=next(iter(uploaded))
    planned=[]
    with zipfile.ZipFile(archive) as z:
        for entry in z.infolist():
            if entry.is_dir(): continue
            parts=PurePosixPath(entry.filename).parts
            if not parts or parts[0] not in ('adaptive-qa','adaptive-confidence-agent') or '..' in parts:
                raise ValueError('Expected a source ZIP with adaptive-qa/ or adaptive-confidence-agent/ at its root')
            rel=PurePosixPath(*parts[1:])
            if not rel.parts: continue
            allowed=rel.parts[0] in ('src','scripts','configs','notebooks') or len(rel.parts)==1
            allowed=allowed or rel.parts[:2]==('data','fixtures')
            if not allowed or '.git' in rel.parts or '__pycache__' in rel.parts or rel.name.startswith('.env'):
                continue
            target=PROJECT/str(rel)
            if target.exists(): raise FileExistsError(f'Existing file: {target}. Preserve it and choose a new working folder.')
            if not target.resolve().is_relative_to(PROJECT.resolve()): raise ValueError('Unsafe archive path')
            planned.append((target,z.read(entry)))
    if not any(str(p).endswith('src/adaptive_qa/pilot.py') for p,_ in planned):
        raise ValueError('This ZIP lacks the current consistency pilot update')
    for target,data in planned:
        target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(data)
else:
    print('Reusing existing Drive source; no ZIP was applied.')
if not (PROJECT/'src/adaptive_qa/pilot.py').exists():
    raise ValueError('Old project found. Preserve it and use a new working folder for this update.')
os.chdir(PROJECT)
print('Working directory:', Path.cwd())


In [ ]:
os.chdir(PROJECT)
subprocess.run([sys.executable,'-m','pip','install','-e','.[gpu,data]'],check=True)


If installation requests a runtime restart, restart and rerun mount, working-directory and installation cells. Do not overwrite existing reviewed files.


In [ ]:
LOGS=PROJECT/'results/logs';LOGS.mkdir(parents=True,exist_ok=True)
def run_script(script,*args):
    """Log each invocation, including failure status and usage snapshots."""
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    folder=LOGS/stamp;folder.mkdir()
    command=[sys.executable,'-u',script,*map(str,args)]
    start=time.time();returncode=None
    try:
        with (folder/'stdout.log').open('w',encoding='utf-8') as log:
            proc=subprocess.Popen(command,cwd=PROJECT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
            for line in proc.stdout:
                print(line,end='');log.write(line);log.flush()
            returncode=proc.wait()
    finally:
        record={'command':command,'started_utc':stamp,'wall_seconds':time.time()-start,'returncode':returncode,
                'note':'A disconnected runtime may prevent final logging. Record interruptions manually too.'}
        (folder/'invocation.json').write_text(json.dumps(record,indent=2)+'\n')
        for path in (PROJECT/'results/runs').glob('*/*usage.json'):
            if path.stat().st_mtime>=start:
                shutil.copy2(path,folder/(path.parent.name+'__'+path.name))
    if returncode!=0: raise RuntimeError(f'Command failed or was interrupted; see {folder}')
    return folder

def pipeline(command,config,execute=False):
    args=[command,'--config',config]
    if execute:
        import torch
        if not torch.cuda.is_available(): raise RuntimeError('A free CUDA GPU is needed for this config; do not buy an upgrade.')
        args.append('--execute-model')
    return run_script('scripts/run.py',*args)


In [ ]:
CONFIG='configs/colab_qanta2021_pilot10.yaml'


## Download and select the public data
These cells can run on CPU. Selection runs only once.


In [ ]:
run_script('scripts/prepare_qanta_public.py','download')
if not (PROJECT/'data/processed/qanta2021/selection.json').exists():
    run_script('scripts/prepare_qanta_public.py','select')
else:
    print('Existing selection preserved.')


## Human answer review
Download the CSV. Read every question and answerline, edit only accepted answer aliases, and mark `approved=yes` only after checking. Use UTF-8 CSV. Do not change source fields or use model predictions to choose aliases.


In [ ]:
review=PROJECT/'data/processed/qanta2021/answer_review.csv'
files.download(str(review))


Run the next cell after editing the CSV. If the selection is already frozen, skip the upload and freeze cells. Do not replace reviewed answers after inference.


In [ ]:
if (PROJECT/'data/processed/qanta2021/frozen_manifest.json').exists():
    raise RuntimeError('Already frozen. Preserve the reviewed selection and skip this upload cell.')
uploaded=files.upload()
if len(uploaded)!=1: raise ValueError('Upload exactly the reviewed answer_review.csv')
name,data=next(iter(uploaded.items()))
if not name.lower().endswith('.csv'): raise ValueError('Expected CSV')
review.write_bytes(data)
run_script('scripts/prepare_qanta_public.py','freeze')


## Build the frozen Wikipedia corpus
Provide your real contact email or project URL for the API request header. Inspect the resulting passages and source records. This uses training titles, never QANTA questions as evidence.


In [ ]:
from getpass import getpass
contact=getpass('Contact email or project URL for Wikipedia: ')
# Avoid recording contact details in the command logger.
subprocess.run([sys.executable,'scripts/build_wikipedia_corpus.py','--contact',contact],check=True)
del contact


## Pin the model and check planning limits
Do this before any inference. Re-running preserves the revision.


In [ ]:
run_script('scripts/pin_colab_model.py')
import yaml
config=yaml.safe_load((PROJECT/CONFIG).read_text())
assert config['calibration']['enabled'] is False
assert config['controller']['actions']==['RETRIEVE'] and config['controller']['max_steps']==1
assert config['model']['samples']==2 and config['evaluation']['include_wait'] is False
print('Pinned model:',config['model']['name'],config['model']['revision'])


In [ ]:
pipeline('prepare',CONFIG)
pipeline('estimate',CONFIG)


## Request a free GPU, then measure the setup
If you switch runtime type, rerun mount/setup/install/logger/config cells. Keep the same Drive files. Read the estimate first: at most 100 unique generation requests in the full ten-question collection.


In [ ]:
import torch, importlib.metadata
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU assigned. Continue CPU work or try a free GPU later.')
info={'timestamp_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
      'python':platform.python_version(),'gpu':torch.cuda.get_device_name(0),
      'vram_gib':torch.cuda.get_device_properties(0).total_memory/2**30,
      'packages':{p:importlib.metadata.version(p) for p in ('torch','transformers','numpy','scikit-learn','PyYAML')}}
stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
(LOGS/f'hardware_{stamp}.json').write_text(json.dumps(info,indent=2)+'\n')
print(json.dumps(info,indent=2))


In [ ]:
pipeline('baseline',CONFIG,execute=True)


Inspect the saved baseline JSONL and initial report. If the model fails to load or mostly produces invalid answers, diagnose that before continuing. A repeated wrong answer is stable but wrong.


In [ ]:
pipeline('collect',CONFIG,execute=True)


In [ ]:
run_script('scripts/verify_collection.py','--config',CONFIG)
pipeline('pilot-report',CONFIG)


In [ ]:
out=PROJECT/'results/runs'/config['experiment_id']
print((out/'pilot_report.md').read_text())
print(json.dumps(json.loads((out/'pilot_report.json').read_text()),indent=2))


## Record your own results and back up
Complete `PROFESSOR_STAGE_02_UPDATE.docx` in your separate private folder using real observations. Include hardware, valid-output rate, errors, logical calls/tokens and actual wall time. Download the pinned pilot10 config, compare it with the local config, and commit the reviewed revision. Keep evidence and reports in your separate private folder. A private backup includes full data/cache/results; keep it out of Git.


In [ ]:
# Save the pinned config to your PC; compare it before updating the local repository.
files.download(str(PROJECT/CONFIG))


In [ ]:
# Keep a second private copy on your PC; do not commit this entire backup to GitHub.
stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
backup=Path('/content')/f'adaptiveqa-private-backup-{stamp}.zip'
with zipfile.ZipFile(backup,'w',zipfile.ZIP_DEFLATED) as z:
    for path in PROJECT.rglob('*'):
        if not path.is_file(): continue
        rel=path.relative_to(PROJECT)
        if any(part in ('.git','.venv','__pycache__','model_cache') for part in rel.parts): continue
        if path.name.startswith('.env') or path.suffix in ('.safetensors','.bin','.pt','.zip'): continue
        if path.name in ('qanta.train.2021.12.20.json','qanta.dev.2021.12.20.json'): continue
        z.write(path,Path('adaptive-confidence-agent')/rel)
print('Backup:',backup,'bytes:',backup.stat().st_size)
files.download(str(backup))
